# CS116 | Stack OOF công khai (bản của nhóm, v1)

Phát triển từ notebook [S6E10 | Stacking 201 public OOF members](https://www.kaggle.com/code/yeashusemwal/secrets-to-satisfaction-it-s-stacking)
(bản tải về: `references/C_ensemble_stacking/secrets-to-satisfaction-it-s-stacking.ipynb`). Phần tải dữ liệu (mục 1) giữ nguyên code gốc;
danh sách nguồn cần ghi công nằm trong notebook gốc.

**Khác bản gốc:**
1. **Bỏ mục 6** (rank-average 2 file submit công khai với trọng số chọn theo LB). Nhóm chọn bài theo nested CV.
2. Kiểm tra fold seed 42 của member trùng `folds.csv` của nhóm (nếu có gắn dataset của nhóm).
3. Chạy 2 biến thể và lưu cả hai:
   - `all`: toàn bộ member, tái tạo bản gốc (LB tác giả báo 0,96181), dùng để **kiểm tra pipeline**.
   - `clean`: bỏ các member là blend/stack của người khác (`stacklevel`, `najiama`), vì OOF của blend có thể lạc quan giả.
4. Lưu `oof_<tên>.npy`, `test_<tên>.npy` theo quy ước repo, và in dòng log cho `experiments.csv`.
5. Có chỗ gắn OOF của nhóm (B, C) làm member (mục 2b), để đo stack tăng bao nhiêu khi thêm model của nhóm.

**Cách chạy trên Kaggle:** mở notebook gốc → *Copy & Edit* (các dataset đầu vào tự gắn theo) → *File → Import Notebook* chọn file này
(hoặc dán từng cell). Không cần GPU, chạy khoảng 15-20 phút trên CPU.

## 1. Tải toàn bộ OOF công khai (code gốc)

In [1]:
import glob, os, time, json
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

T0 = time.time()
def find_dir(slug):
    hits = [p for p in glob.glob(f'/kaggle/input/**/{slug}', recursive=True) if os.path.isdir(p)]
    assert hits, f'input not attached: {slug}'
    return sorted(hits, key=len)[0]

COMP = find_dir('playground-series-s6e10')
tr, te = pd.read_csv(f'{COMP}/train.csv'), pd.read_csv(f'{COMP}/test.csv')
y = tr['satisfaction'].astype(int).values
NTR = len(y)
OOF, TEST, SRC = {}, {}, {}

def add(name, o, t, src):
    o, t = np.asarray(o, float), np.asarray(t, float)
    assert o.shape == (NTR,) and t.shape == (len(te),), (name, o.shape, t.shape)
    assert np.isfinite(o).all() and np.isfinite(t).all(), name
    OOF[name], TEST[name], SRC[name] = o, t, src

def by_id(df, ids, col):
    return df.set_index('id')[col].loc[ids].values

In [2]:
# ---------------- kratosyan's 124-member set ----------------
def l_gp():   # Part 8 members; that notebook's output now holds Part 9, so they are read from Chris Deotte's 124-member dataset
    d = find_dir('s6e10-124-public-oof-and-test-predictions')
    for f in sorted(glob.glob(f'{d}/**/oof_gp_*.npy', recursive=True)):
        o, t = np.load(f), np.load(f.replace('/oof_gp_', '/test_gp_'))
        assert len(o) == NTR and len(t) == len(te), f
        add(os.path.basename(f)[4:-4], o, t, 'goodpjw P8')
def l_mg():
    d = find_dir('s6e10-oof-library')
    mo, mt = pd.read_parquet(f'{d}/oof_predictions.parquet'), pd.read_parquet(f'{d}/test_predictions.parquet')
    assert (mo['id'].values == tr['id'].values).all() and (mt['id'].values == te['id'].values).all()
    groups = {}
    for c in mo.columns:
        if c in ('id', 'fold', 'satisfaction'):
            continue
        groups.setdefault(c.rsplit('_s', 1)[0] if c.startswith('realmlp') else c, []).append(c)
    for g, cols in groups.items():
        add('mg_' + g, mo[cols].mean(1).values, mt[cols].mean(1).values, 'megayak')
def l_gl():
    d = find_dir('s6e10-golem-oof-library')
    for k in 'abcdefghijklmn':
        add('gl_' + k, np.load(f'{d}/oof_{k}.npy'), np.load(f'{d}/test_{k}.npy'), 'golem')
    for k in ['o_tabpfn35', 'p_lgbm_auxev', 'q_lgbm_auxev_10fold']:   # v3: added to the library on 2026-10-08
        add('gl_' + k, np.load(f'{d}/oof_{k}.npy'), np.load(f'{d}/test_{k}.npy'), 'golem')
def l_ar():
    d = find_dir('s6e10-oof-test-predictions')
    ao, at = pd.read_parquet(f'{d}/oof_predictions.parquet'), pd.read_parquet(f'{d}/test_predictions.parquet')
    assert (ao['id'].values == tr['id'].values).all() and (at['id'].values == te['id'].values).all()
    for c in ['lgb_5f', 'xgb_5f', 'cat_5f', 'realmlp_5f']:
        add('ar_' + c, ao[c].values, at[c].values, 'arhancanli12')
def l_sa():
    d = find_dir('s6e10-stack-oof-predictions')
    for f in sorted(glob.glob(f'{d}/*.npz')):
        z = np.load(f)
        add('sa_' + os.path.basename(f)[:-4], z['oof'], z['test'], 'sachith7')
def l_nj():
    d = find_dir('s6e10-oof')
    add('nj_01', by_id(pd.read_csv(f'{d}/01_blend_oof.csv'), tr['id'], 'satisfaction'),
        by_id(pd.read_csv(f'{d}/01_submission.csv'), te['id'], 'satisfaction'), 'najiama')
def l_pfn():
    for nm, slug in [('pfn_samanyu_route_og', 's6e10-tabpfn-route-og'), ('pfn_samanyu_seeds', 's6e10-tabpfn-seeds'),
                     ('pfn_hemingweb_raw', 's6e10-exp01-tabpfn35-full-context')]:
        d = find_dir(slug)
        add(nm, by_id(pd.read_csv(f'{d}/oof_exp01_tabpfn35.csv'), tr['id'], 'pred'),
            by_id(pd.read_csv(f'{d}/test_exp01_tabpfn35.csv'), te['id'], 'satisfaction'), 'TabPFN outputs')
def l_mi():
    d = find_dir('s6e10-analysis-first-equality-blocks-stack')
    mo2 = pd.read_parquet(f'{d}/oof_predictions.parquet').set_index('id').loc[tr['id']]
    mt2 = pd.read_parquet(f'{d}/test_predictions.parquet').set_index('id').loc[te['id']]
    for c in mo2.columns:
        add('mi_' + c, mo2[c].values, mt2[c].values, 'mitudru')
def l_wx():
    for slug, c in [('s6e10-catboost-rating-crosses-cv-0-96080', 'rating_segments'),
                    ('s6e10-lightgbm-context-stats-cv-0-96065', 'context_numeric')]:
        d = find_dir(slug)
        add('wx_' + c, by_id(pd.read_parquet(f'{d}/oof_predictions.parquet'), tr['id'], c),
            by_id(pd.read_parquet(f'{d}/test_predictions.parquet'), te['id'], c), 'wangxintong111')
def l_th():
    for slug, tag in [('s6e10-xgb-realmlp-catboost-lb-0-96123', 'thisray'), ('s6e10-honest-oof-stack', 'megayak honest')]:
        d = find_dir(slug)
        o = pd.read_csv(f'{d}/oof_members.csv').set_index('id').loc[tr['id']]
        t = pd.read_csv(f'{d}/test_members.csv').set_index('id').loc[te['id']]
        for c in t.columns:
            add(f'{tag.split()[0][:2]}{"h" if "honest" in tag else ""}_{c}', o[c].values, t[c].values, tag)
def l_bp():
    d = find_dir('s6e10-busyaprime-five-members-oof')
    for f in sorted(glob.glob(f'{d}/bp_*.npz')):
        z = np.load(f)
        add(os.path.basename(f)[:-4], z['oof'], z['test'], 'busyaprime')

# ---------------- new members (not in kratosyan v3) ----------------
def l_ab():  # abdullahsafwan333: 15-fold x 3-seed TabPFN-3.5 variants + CatBoost/XGB (15 folds)
    d = find_dir('oof-dataset')
    for m in ['tabpfn_catfd10', 'tabpfn_raw', 'tabpfn_te_cnt', 'cat_v3', 'xgb_v3']:
        add('ab_' + m, by_id(pd.read_csv(f'{d}/{m}_oof.csv'), tr['id'], 'pred'),
            by_id(pd.read_csv(f'{d}/{m}_test.csv'), te['id'], 'pred'), 'abdullah ' + ('pfn' if 'tabpfn' in m else 'gbdt'))
def l_kx():  # kratosyan 10-fold route + original-teacher + aux members
    d = find_dir('s6e10-xgboost-realmlp-route-id-teacher')
    for f in sorted(glob.glob(f'{d}/**/*_route_ogaux.npz', recursive=True)):
        z = np.load(f)
        add('kx_' + os.path.basename(f)[:-4], z['oof'], z['test'], 'kratosyan10f')
def l_hg():
    d = find_dir('single-catboost-eda-ps6e10')
    add('hg_cat', np.load(glob.glob(f'{d}/**/oof_train.npy', recursive=True)[0]),
        np.load(glob.glob(f'{d}/**/oof_test.npy', recursive=True)[0]), 'hermengardo')
def l_dp():
    d = find_dir('s6e10-numeric-keys-catboost-ctr')
    o = pd.read_csv(glob.glob(f'{d}/**/oof_predictions.csv', recursive=True)[0])
    tp = np.zeros(len(te)); n = 0
    for f in sorted(glob.glob(f'{d}/**/catboost_numeric_ctr_fold*.npz', recursive=True)):
        z = np.load(f)
        assert (z['test_ids'] == te['id'].values).all()
        tp += z['test_pred']; n += 1
    assert n == 5, n
    add('dp_catctr', by_id(o, tr['id'], 'catboost_numeric_ctr'), tp / n, 'denpugovkin')
def l_yk():
    d = find_dir('ps-s6-e10-realmlp-pytabkit')
    add('yk_realmlp', by_id(pd.read_csv(f'{d}/oof_preds.csv'), tr['id'], 'satisfaction'),
        by_id(pd.read_csv(f'{d}/submission.csv'), te['id'], 'satisfaction'), 'yekenot')


# Sadam Torres artifacts (38 members + 2 original-data teacher predictions)
def l_st():
    d = find_dir('ps-s6e10-airline-satisfaction-artifacts')
    for f in sorted(glob.glob(f'{d}/**/preds_*.npz', recursive=True)):
        z = np.load(f)
        add('st_' + os.path.basename(f)[6:-4], z['oof'], z['test'], 'sadamtorres')
    for f in ['orig_cat_pred', 'orig_model_pred']:
        z = np.load(glob.glob(f'{d}/**/{f}.npz', recursive=True)[0])
        add('st_' + f, z['train'], z['test'], 'orig teacher')
# Aman Atar original-prior stack v5 (5 base members)
def l_am():
    d = find_dir('s6e10-the-original-prior-stack-v5')
    z = np.load(glob.glob(f'{d}/**/oof_artifacts.npz', recursive=True)[0])
    for k in ['lgbm_champion', 'lgbm_te', 'lgbm_te_rows', 'xgb_gpu', 'catboost_gpu']:
        add('am_' + k, z['oof_' + k], z['test_' + k], 'amanatar')
# Ahmet Kagan Koral (5 members)
def l_kk():
    d = find_dir('s6e10-encodings-catboost-realmlp-lb-0-961')
    o, t = pd.read_csv(f'{d}/oof_members.csv'), pd.read_csv(f'{d}/test_members.csv')
    assert (o['id'].values == tr['id'].values).all() and (t['id'].values == te['id'].values).all()
    for c in o.columns[1:]:
        add('kk_' + c, o[c].values, t[c].values, 'kagankoral')
# stack-level OOFs: busyaprime's stack (LB 0.96169) and najiama blends 02/03
def l_stacklevel():
    d = find_dir('s6e10-route-or-distance-both-lb-0-96169')
    z = np.load(glob.glob(f'{d}/**/oof_stacks.npz', recursive=True)[0])
    add('bp_stack53', z['oof53'], z['test53'], 'stacklevel')
    d = find_dir('s6e10-oof')
    for k in ['02', '03']:
        add('nj_' + k, by_id(pd.read_csv(f'{d}/{k}_blend_oof.csv'), tr['id'], 'satisfaction'),
            by_id(pd.read_csv(f'{d}/{k}_submission.csv'), te['id'], 'satisfaction'), 'stacklevel')

# Chris Deotte: 9 tabular foundation models + tuned XGBoost / RealMLP (SKF5 rs42, train/test row order)
def l_cd():
    d = find_dir('s6e10-tfm-oof-and-test-predictions')
    for f in sorted(glob.glob(f'{d}/**/oof_*.npy', recursive=True)):
        m = os.path.basename(f)[4:-4]
        add('cd_' + m, np.load(f), np.load(f.replace('oof_', 'test_')), 'cdeotte tfm')

# Every source is optional: if an author re-runs or removes a file, that library is skipped (and reported) instead of
# breaking the notebook. Kernel outputs are attached at their latest version, so the member count can change over time.
SKIPPED = []
for fn in [l_gp, l_mg, l_gl, l_ar, l_sa, l_nj, l_pfn, l_mi, l_wx, l_th, l_bp, l_ab, l_kx, l_hg, l_dp, l_yk,
           l_st, l_am, l_kk, l_stacklevel, l_cd]:
    before = dict(OOF)
    try:
        fn()
    except Exception as e:
        for k in set(OOF) - set(before):   # roll back a partially loaded library
            OOF.pop(k); TEST.pop(k); SRC.pop(k)
        SKIPPED.append(fn.__name__)
        print(f'skipped {fn.__name__}: {type(e).__name__}: {e}')
names = list(OOF)
print(len(names), 'members |', pd.Series(SRC).value_counts().to_dict())
print(f'loaded in {time.time() - T0:.0f}s; skipped libraries: {SKIPPED}')

skipped l_dp: AssertionError: input not attached: s6e10-numeric-keys-catboost-ctr
202 members | {'goodpjw P8': 48, 'sadamtorres': 38, 'sachith7': 24, 'golem': 17, 'megayak': 11, 'cdeotte tfm': 11, 'mitudru': 7, 'kagankoral': 5, 'amanatar': 5, 'busyaprime': 5, 'arhancanli12': 4, 'megayak honest': 4, 'TabPFN outputs': 3, 'stacklevel': 3, 'thisray': 3, 'abdullah pfn': 3, 'kratosyan10f': 2, 'wangxintong111': 2, 'abdullah gbdt': 2, 'orig teacher': 2, 'najiama': 1, 'hermengardo': 1, 'yekenot': 1}
loaded in 34s; skipped libraries: ['l_dp']


## 2. Kiểm tra fold và thêm OOF của nhóm

In [3]:
from pathlib import Path

# Fold mà gần như mọi member công khai dùng: StratifiedKFold(5, shuffle=True, random_state=42)
fold42 = np.zeros(NTR, int)
for k, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(NTR), y)):
    fold42[va] = k

# Nếu đã gắn dataset của nhóm (chứa folds.csv và oof_*.npy), kiểm tra trùng fold
TEAM_SLUG = 'cs116-s6e10-oof'   # đổi theo tên dataset nhóm tạo trên Kaggle
team_dirs = [p for p in glob.glob(f'/kaggle/input/**/{TEAM_SLUG}', recursive=True) if os.path.isdir(p)]
TEAM_DIR = Path(sorted(team_dirs, key=len)[0]) if team_dirs else None
if TEAM_DIR is not None and (TEAM_DIR / 'folds.csv').exists():
    f = pd.read_csv(TEAM_DIR / 'folds.csv').set_index('id').loc[tr['id'], 'fold'].values
    assert (f == fold42).all(), 'folds.csv của nhóm KHÁC fold seed 42!'
    print('folds.csv của nhóm trùng 100% với fold seed 42')
else:
    print('Chưa gắn dataset của nhóm: bỏ qua kiểm tra folds.csv (đã kiểm tra trên máy: trùng 100%)')

folds.csv của nhóm trùng 100% với fold seed 42


### 2b. OOF của nhóm (tùy chọn)
Mỗi model của nhóm lưu `oof_<tên>.npy` + `test_<tên>.npy` (thứ tự dòng như `train.csv` / `test.csv`, fold theo `folds.csv`).
Upload thư mục `oof/` thành một Kaggle dataset rồi gắn vào notebook này. Member của nhóm có tiền tố `team_`.

In [4]:
TEAM_MEMBERS = []
if TEAM_DIR is not None:
    for fo in sorted(TEAM_DIR.glob('oof_*.npy')):
        m = fo.stem[4:]
        ft = TEAM_DIR / f'test_{m}.npy'
        if not ft.exists():
            print(f'bỏ qua {m}: thiếu test_{m}.npy'); continue
        o, t = np.load(fo), np.load(ft)
        if roc_auc_score(y, o) < 0.95:   # loại baseline yếu (vd logreg 0.92), không giúp stack
            print(f'bỏ qua {m}: OOF AUC {roc_auc_score(y, o):.5f} < 0.95'); continue
        add('team_' + m, o, t, 'team'); TEAM_MEMBERS.append('team_' + m)
names = list(OOF)
print(f'{len(names)} member, trong đó của nhóm: {TEAM_MEMBERS}')

204 member, trong đó của nhóm: ['team_A_lgb_v1', 'team_A_lr_v1']


In [5]:
single = pd.Series({n: roc_auc_score(y, OOF[n]) for n in names}).sort_values(ascending=False)
best = single.groupby(pd.Series(SRC)).agg(['size', 'max']).sort_values('max', ascending=False)
best.columns = ['members', 'best single OOF AUC']
best

,members,best single OOF AUC
stacklevel,3,0.961989
najiama,1,0.961744
abdullah pfn,3,0.961596
sachith7,24,0.961413
abdullah gbdt,2,0.961396
golem,17,0.961388
goodpjw P8,48,0.961345
mitudru,7,0.961307
TabPFN outputs,3,0.961268
sadamtorres,38,0.961253


## 3. Nested LR stack trên logit
Giống bản gốc: `LogisticRegression(C=0.003)`, meta fold `StratifiedKFold(5, shuffle=True, random_state=7)` (khác seed 42 của member,
để stacker được chấm trên dòng nó chưa thấy). Các biến thể:
- `all`: mọi member; `clean`: bỏ member là blend/stack của người khác;
- `clean_public` (khi có member nhóm): `clean` bỏ member nhóm. OOF của nó dùng để phân tích phần sai số trên máy.

Ma trận logit tính một lần (`Z`, `ZT`), mỗi stack chỉ chọn cột: bản trước tạo lại ma trận mỗi lần nên hết RAM.
Mỗi member được đưa về thang logit theo đúng dạng lưu (`to_logit`, giống `src/blend.py`): khoảng 9 member công khai
lưu sẵn dạng logit/điểm và vài member dạng rank; bản trước clip tất cả như xác suất nên các cột đó bị cắt méo.
Mỗi stack lưu `oof_stack_lr_<biến thể>.npy` / `test_...` ngay sau khi chạy xong.

In [6]:
import gc
from scipy.stats import rankdata

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def is_rank(p):
    """Giá trị trải đều trên (0, 1] như rankdata(p) / len(p)."""
    q = np.quantile(p, [0.1, 0.25, 0.5, 0.75, 0.9])
    return bool(p.min() >= 0 and p.max() <= 1 and np.abs(q - [0.1, 0.25, 0.5, 0.75, 0.9]).max() < 0.01)

def to_logit(o, t):
    """Đưa member về thang logit như src/blend.py: logit giữ nguyên, rank -> logit(rank), xác suất -> logit.
    Bản cũ clip mọi cột về [1e-6, 1-1e-6] nên cột lưu sẵn dạng logit/điểm (vd ar_*, bp_stack53) bị cắt méo."""
    if min(o.min(), t.min()) < 0 or max(o.max(), t.max()) > 1:
        return np.clip(o, -30, 30), np.clip(t, -30, 30), 'logit'
    if is_rank(o):
        lr = lambda p: logit(rankdata(p) / (len(p) + 1))
        return lr(o), lr(t), 'rank'
    return logit(o), logit(t), 'prob'

# tính logit một lần cho mọi member rồi bỏ dict xác suất để nhẹ RAM
COL = {n: i for i, n in enumerate(names)}
Z, ZT, SCALE = np.empty((NTR, len(names))), np.empty((len(te), len(names))), {}
for i, n in enumerate(names):
    Z[:, i], ZT[:, i], SCALE[n] = to_logit(OOF[n], TEST[n])
OOF.clear(); TEST.clear(); gc.collect()
print('thang đo của member:', pd.Series(SCALE).value_counts().to_dict(),
      '| không phải xác suất:', [n for n, s in SCALE.items() if s != 'prob'])
print(f'Z {Z.shape}, ZT {ZT.shape}: {(Z.nbytes + ZT.nbytes) / 1e9:.1f} GB')

META = list(StratifiedKFold(5, shuffle=True, random_state=7).split(np.zeros(NTR), y))
def report(tag, o):
    print(f'{tag:28s} nested CV {roc_auc_score(y, o):.6f}  folds ' + ' '.join(f'{roc_auc_score(y[b], o[b]):.5f}' for _, b in META), flush=True)

def stack_oof(cols, meta, C=0.003):
    idx = [COL[n] for n in cols]
    o = np.zeros(NTR)
    for a, b in meta:
        m = LogisticRegression(C=C, max_iter=3000).fit(Z[np.ix_(a, idx)], y[a])
        o[b] = m.decision_function(Z[np.ix_(b, idx)])
        del m; gc.collect()
    return o

def lr_stack(cols, C=0.003):
    idx = [COL[n] for n in cols]
    test = LogisticRegression(C=C, max_iter=3000).fit(Z[:, idx], y).decision_function(ZT[:, idx])
    gc.collect()
    return stack_oof(cols, META, C), test

sig = lambda v: 1 / (1 + np.exp(-v))
BLEND_SRC = {'stacklevel', 'najiama'}   # member là blend/stack của người khác
SETS = {'all': names, 'clean': [n for n in names if SRC[n] not in BLEND_SRC]}
if TEAM_MEMBERS:
    SETS['clean_public'] = [n for n in SETS['clean'] if n not in TEAM_MEMBERS]
RES = {}
for tag, cols in SETS.items():
    t = time.time()
    RES[tag] = lr_stack(cols)
    np.save(f'oof_stack_lr_{tag}.npy', sig(RES[tag][0])); np.save(f'test_stack_lr_{tag}.npy', sig(RES[tag][1]))
    report(f'LR stack {tag} ({len(cols)})', RES[tag][0]); print(f'({time.time() - t:.0f}s, đã lưu)')

thang đo của member: {'prob': 196, 'logit': 5, 'rank': 3} | không phải xác suất: ['ar_lgb_5f', 'ar_xgb_5f', 'ar_cat_5f', 'ar_realmlp_5f', 'nj_01', 'bp_stack53', 'nj_02', 'nj_03']
Z (699635, 204), ZT (299844, 204): 1.6 GB
LR stack all (204)           nested CV 0.962172  folds 0.96283 0.96170 0.96188 0.96209 0.96238
(178s, đã lưu)
LR stack clean (200)         nested CV 0.962170  folds 0.96282 0.96170 0.96188 0.96209 0.96237
(156s, đã lưu)
LR stack clean_public (198)  nested CV 0.962170  folds 0.96282 0.96169 0.96188 0.96209 0.96238
(150s, đã lưu)


## 4. So 5 cấu hình stack trên cùng meta fold (3 seed)
| | Cấu hình | Kiểm tra |
|---|---|---|
| A | pool công khai `clean_public` | nền |
| B | A + member nhóm | member nhóm có giúp không |
| C | A đã bỏ gần trùng (quy tắc cố định: tương quan logit > 0,999 → giữ member AUC cao hơn) | bỏ trùng có đổi gì không |
| D | C + member nhóm | bỏ trùng có "nhường chỗ" cho member nhóm không |
| E | A, stacker riêng cho từng giá trị `Type of Travel` | 2 nhóm khách AUC rất khác (Business ~0,955, Personal ~0,828) |

Mức chênh dưới khoảng 0,00001 (10 x 1e-6) xem như nhiễu; một cấu hình đáng dùng khi hơn A ở cả 3 seed và phần lớn fold.
Quy tắc bỏ trùng đặt trước, không chọn theo CV (tránh bẫy ở docs/STACKING.md mục 8).

In [7]:
def compare(o_new, o_old, meta):
    d = roc_auc_score(y, o_new) - roc_auc_score(y, o_old)
    up = sum(roc_auc_score(y[b], o_new[b]) > roc_auc_score(y[b], o_old[b]) for _, b in meta)
    return f'{d * 1e6:+7.1f} x 1e-6, tăng {up}/5 fold'

def dedup(cols, thr=0.999, n_rows=200_000):
    """Duyệt từ AUC cao xuống, bỏ member có tương quan logit > thr với một member đã giữ (tính trên 200k dòng)."""
    rows = np.sort(np.random.default_rng(0).choice(NTR, min(n_rows, NTR), replace=False))
    order = sorted(cols, key=lambda n: -single[n])
    M = Z[np.ix_(rows, [COL[n] for n in order])].astype(np.float32)
    M = (M - M.mean(0)) / M.std(0)
    R = (M.T @ M) / len(rows)
    kept = []
    for i in range(len(order)):
        if not any(R[i, j] > thr for j in kept):
            kept.append(i)
    return [order[i] for i in kept]

def stack_oof_seg(cols, meta, C=0.003):
    """Stacker riêng cho từng giá trị Type of Travel (cùng meta fold)."""
    idx, seg = [COL[n] for n in cols], tr['Type of Travel'].values
    o = np.zeros(NTR)
    for a, b in meta:
        for s in np.unique(seg):
            aa, bb = a[seg[a] == s], b[seg[b] == s]
            m = LogisticRegression(C=C, max_iter=3000).fit(Z[np.ix_(aa, idx)], y[aa])
            o[bb] = m.decision_function(Z[np.ix_(bb, idx)])
        gc.collect()
    return o

PUB = SETS.get('clean_public', SETS['clean'])
PUB_DEDUP = dedup(PUB)
print(f'bỏ gần trùng: {len(PUB)} -> {len(PUB_DEDUP)} member')
CONFIGS = {'A': (PUB, stack_oof), 'C': (PUB_DEDUP, stack_oof), 'E': (PUB, stack_oof_seg)}
if TEAM_MEMBERS:
    CONFIGS.update({'B': (PUB + TEAM_MEMBERS, stack_oof), 'D': (PUB_DEDUP + TEAM_MEMBERS, stack_oof)})

# bản nộp được của C / D (meta seed 7 + test fit trên toàn bộ OOF), lưu như mục 3
for tag, cols in [('dedup', PUB_DEDUP)] + ([('dedup_team', PUB_DEDUP + TEAM_MEMBERS)] if TEAM_MEMBERS else []):
    t = time.time()
    RES[tag] = lr_stack(cols); SETS[tag] = cols
    np.save(f'oof_stack_lr_{tag}.npy', sig(RES[tag][0])); np.save(f'test_stack_lr_{tag}.npy', sig(RES[tag][1]))
    report(f'LR stack {tag} ({len(cols)})', RES[tag][0]); print(f'({time.time() - t:.0f}s, đã lưu)')

SEED7 = {'A': RES.get('clean_public', RES['clean'])[0], 'C': RES['dedup'][0]}
if TEAM_MEMBERS:
    SEED7.update({'B': RES['clean'][0], 'D': RES['dedup_team'][0]})
for seed in [7, 101, 2026]:
    t = time.time()
    meta = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(NTR), y))
    out = {}
    for tag, (cols, fn) in CONFIGS.items():
        out[tag] = SEED7[tag] if (seed == 7 and tag in SEED7) else fn(cols, meta)
    print(f'meta seed {seed}: A (nền) {roc_auc_score(y, out["A"]):.6f}')
    for tag in [x for x in 'BCDE' if x in out]:
        print(f'  {tag} vs A: {compare(out[tag], out["A"], meta)}   ({roc_auc_score(y, out[tag]):.6f})')
    if 'D' in out:
        print(f'  D vs C (member nhóm trên pool đã bỏ trùng): {compare(out["D"], out["C"], meta)}')
    print(f'  ({time.time() - t:.0f}s)', flush=True)
    gc.collect()

bỏ gần trùng: 198 -> 153 member
LR stack dedup (153)         nested CV 0.962168  folds 0.96281 0.96170 0.96187 0.96208 0.96238
(152s, đã lưu)
LR stack dedup_team (155)    nested CV 0.962168  folds 0.96281 0.96170 0.96187 0.96208 0.96238
(124s, đã lưu)
meta seed 7: A (nền) 0.962170
  B vs A:    -0.2 x 1e-6, tăng 2/5 fold   (0.962170)
  C vs A:    -1.9 x 1e-6, tăng 2/5 fold   (0.962168)
  D vs A:    -1.7 x 1e-6, tăng 1/5 fold   (0.962168)
  E vs A:    -7.1 x 1e-6, tăng 2/5 fold   (0.962163)
  D vs C (member nhóm trên pool đã bỏ trùng):    +0.2 x 1e-6, tăng 2/5 fold
  (109s)
meta seed 101: A (nền) 0.962171
  B vs A:    +2.2 x 1e-6, tăng 4/5 fold   (0.962173)
  C vs A:    -3.9 x 1e-6, tăng 2/5 fold   (0.962167)
  D vs A:    -1.3 x 1e-6, tăng 2/5 fold   (0.962169)
  E vs A:    -9.6 x 1e-6, tăng 2/5 fold   (0.962161)
  D vs C (member nhóm trên pool đã bỏ trùng):    +2.7 x 1e-6, tăng 4/5 fold
  (622s)
meta seed 2026: A (nền) 0.962179
  B vs A:    +1.9 x 1e-6, tăng 3/5 fold   (0.962181)
  C vs

## 5. Lưu kết quả
- `submission.csv` = biến thể `all` (tái tạo bản gốc, để so LB với 0,96181 và xác nhận pipeline đúng).
- `sub_stack_lr_clean.csv` = biến thể `clean`.
- `oof_stack_lr_<biến thể>.npy`, `test_stack_lr_<biến thể>.npy` (xác suất) để đưa về repo / blend tiếp.

In [8]:
SUBMIT = 'all'   # đổi thành 'clean' để nộp biến thể clean làm submission.csv
log_rows = []
for tag, (o, t) in RES.items():   # file .npy đã lưu ở mục 3
    t_p = sig(t)
    assert t_p.shape == (len(te),) and np.isfinite(t_p).all()
    sub = pd.DataFrame({'id': te['id'], 'satisfaction': t_p})
    sub.to_csv(f'sub_stack_lr_{tag}.csv', index=False)
    if tag == SUBMIT:
        sub.to_csv('submission.csv', index=False)
    log_rows.append(f"{time.strftime('%Y-%m-%d')},A,stack_lr_{tag},{roc_auc_score(y, o):.6f},,"
                    f"nested CV (meta seed 7); LR C=0.003 tren {len(SETS[tag])} OOF; tu notebook secrets-to-satisfaction")
print('submission.csv =', SUBMIT)
print('\nDòng cho experiments.csv (điền LB sau khi nộp):')
print('\n'.join(log_rows))
print(f'\ntổng thời gian {time.time() - T0:.0f}s')

submission.csv = all

Dòng cho experiments.csv (điền LB sau khi nộp):
2026-10-15,A,stack_lr_all,0.962172,,nested CV (meta seed 7); LR C=0.003 tren 204 OOF; tu notebook secrets-to-satisfaction
2026-10-15,A,stack_lr_clean,0.962170,,nested CV (meta seed 7); LR C=0.003 tren 200 OOF; tu notebook secrets-to-satisfaction
2026-10-15,A,stack_lr_clean_public,0.962170,,nested CV (meta seed 7); LR C=0.003 tren 198 OOF; tu notebook secrets-to-satisfaction
2026-10-15,A,stack_lr_dedup,0.962168,,nested CV (meta seed 7); LR C=0.003 tren 153 OOF; tu notebook secrets-to-satisfaction
2026-10-15,A,stack_lr_dedup_team,0.962168,,nested CV (meta seed 7); LR C=0.003 tren 155 OOF; tu notebook secrets-to-satisfaction

tổng thời gian 2188s
